# Wessex treated effluent volume forecasting — QA v3 development

与现有代码配套的 **CPU / Colab 续跑入口**。先两个既定开发 SiteID 检查，再 10 个开发 SiteID、三种子。源训练止于 2021，开发止于 2023；本 notebook 不评价 2024–2025，不更换测点，不拟合区间，不制作 App。

使用发布后 Good QA 的历史回顾实验；在线 QA 可用性和独立物理设施关系未验证。3 天预测为第 3 天当天体积。现有结果和失败日志保留。

非 Drive 核心执行已在本地临时项目验证；**没有声称真实 Colab/Drive 挂载已验证，也没有 GPU 执行支持**。详细说明见 `notebooks/README_STAGE2.md`。

In [ ]:
# Colab 交互授权挂载；本地 Jupyter 可跳过此单元。
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
# 全部源模型、目标 checkpoint、预测和日志持久保存在此项目内。
ROOT = Path('/content/drive/MyDrive/Wastewater_Paper_stage2').resolve()
# 如已解压完整项目，保留空列表。代码、数据、source/runs 分包合并到同一 ROOT。
ZIP_PATHS = []  # 例如 [Path('/content/drive/MyDrive/Wastewater_Stage2_Code.zip')]
# 可填发布的 ZIP SHA256；包内校验清单也会自动检查。
EXPECTED_ZIP_SHA256 = {}  # {'Wastewater_Stage2_Code.zip': '...'}
INSTALL_PINNED = False
STRICT_VERSIONS = True


## 安全导入及核心函数

不调用 GitHub、邮箱或公开存储。原全量 raw 不需要：交付的 13 个 SiteID v3 派生 CSV 与审计 manifest 已足够运行本轮。ZIP 不覆盖不同内容；CRC/校验清单不能替代可信来源。

In [ ]:
import collections
import hashlib
import importlib
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import zipfile
from datetime import datetime, timezone

EXPECTED_VERSIONS = {'torch': '2.8.0+cpu', 'numpy': '2.3.5', 'pandas': '2.2.3',
                     'scipy': '1.17.0', 'sklearn': '1.8.0', 'matplotlib': '3.10.8'}


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def inside(root, path):
    root = Path(root).resolve()
    result = (root / path).resolve()
    if not result.is_relative_to(root):
        raise ValueError(f'Path escapes project: {path}')
    return result


def import_zip(archive, root, expected_sha256=None):
    archive, root = Path(archive), Path(root).resolve()
    if expected_sha256 and digest(archive) != expected_sha256:
        raise ValueError('ZIP SHA256 mismatch')
    root.mkdir(parents=True, exist_ok=True)
    stage_parent = root / '.cache' / 'zip_import'
    stage_parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:
            raise ValueError('ZIP CRC test failed')
        entries = [i for i in z.infolist() if not i.is_dir()]
        names = [inside(root, i.filename).relative_to(root).as_posix() for i in entries]
        if len(names) != len(set(names)):
            raise ValueError('Duplicate ZIP member path')
        for info in entries:
            if (chr(92) in info.filename or Path(info.filename).is_absolute()
                or '..' in Path(info.filename).parts):
                raise ValueError('Nonportable ZIP member path')
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError('ZIP symlink rejected')
            inside(root, info.filename)
        with tempfile.TemporaryDirectory(dir=stage_parent) as stage:
            stage = Path(stage)
            for info in entries:
                target = inside(stage, info.filename)
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
            checksum_files = [p for p in stage.rglob('*') if p.is_file() and p.name in
                              ('SHA256SUMS', 'SHA256SUMS.txt')]
            checksum_count = 0
            for checksums in checksum_files:
                for line in checksums.read_text().splitlines():
                    if not line.strip():
                        continue
                    expected, name = line.split(maxsplit=1)
                    name = name.lstrip('*')
                    candidate = inside(checksums.parent, name)
                    if len(expected) != 64 or not candidate.is_file() or digest(candidate) != expected:
                        raise ValueError(f'Package member SHA256 mismatch: {name}')
                    checksum_count += 1
            # Package-specific checksum files validate their archive; they are not
            # merged into ROOT, where another package may have a different list.
            checksum_names = {p.relative_to(stage).as_posix() for p in checksum_files}
            payload_entries = [i for i in entries if
                               inside(stage, i.filename).relative_to(stage).as_posix() not in checksum_names]
            # Check every collision BEFORE copying any payload.
            for info in payload_entries:
                destination = inside(root, info.filename)
                if destination.exists() and (not destination.is_file() or
                    digest(destination) != digest(stage / info.filename)):
                    raise ValueError(f'Existing different file preserved: {info.filename}')
            copied = identical = 0
            for info in payload_entries:
                destination = inside(root, info.filename)
                if destination.exists():
                    identical += 1
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(stage / info.filename, destination)
                copied += 1
    return {'archive': str(archive), 'sha256': digest(archive), 'crc_passed': True,
            'new_files': copied, 'identical_files_preserved': identical,
            'manifest_entries_checked': checksum_count}


def runtime_versions():
    return {key: importlib.import_module(key).__version__ for key in EXPECTED_VERSIONS}


def project_env(root):
    root = Path(root).resolve()
    cache = root / '.cache'
    cache.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = str(root / 'src')
    env['MPLCONFIGDIR'] = str(cache / 'matplotlib')
    env['XDG_CACHE_HOME'] = str(cache)
    env['PYTHONHASHSEED'] = '0'
    return env


def preflight(root, config_name, strict_versions=True):
    root = Path(root).resolve()
    path = inside(root, config_name)
    cfg = json.loads(path.read_text())
    required = ['src/wastewater/shared_data.py', 'src/wastewater/shared_models.py',
                'src/wastewater/shared_experiment.py', cfg['canonical_csv'], cfg['mapping_csv'],
                'data/provenance/DEVELOPMENT_SELECTION_20261009.json',
                'data/provenance/reconciliation_20261009/coverage_2016_2021.csv']
    required.append(str(Path(cfg['canonical_csv']).parent / 'manifest.json'))
    missing = [p for p in required if not inside(root, p).is_file()]
    if missing:
        raise FileNotFoundError('Required project files missing: ' + ', '.join(missing))
    for key in ['canonical_csv', 'mapping_csv', 'source_output', 'output']:
        inside(root, cfg[key])
    env = project_env(root)
    os.environ['MPLCONFIGDIR'] = env['MPLCONFIGDIR']
    os.environ['XDG_CACHE_HOME'] = env['XDG_CACHE_HOME']
    versions = runtime_versions()
    if strict_versions and versions != EXPECTED_VERSIONS:
        raise ValueError(f'Dependency versions differ; do not mix saved runs: {versions}')
    if strict_versions and sys.version_info[:2] != (3, 12):
        raise ValueError('The verified environment uses Python 3.12')
    # Fresh interpreter prevents stale imported project modules after notebook edits.
    snippet = "import json, sys\nfrom pathlib import Path\nimport pandas as pd\nfrom wastewater.shared_experiment import validate, execution_signature\nroot = Path(sys.argv[1]); cfg = json.loads((root / sys.argv[2]).read_text())\nvalidate(cfg, root)\nsignature, provenance = execution_signature(root, cfg)\nsaved_source = root / cfg['source_output'] / 'selection.json'\nif saved_source.exists() and json.loads(saved_source.read_text())['signature'] != signature:\n    raise ValueError('Source signature changed; new version directory required')\nsaved_target = root / cfg['output'] / 'manifest.json'\nif saved_target.exists():\n    prior = json.loads(saved_target.read_text())\n    from wastewater.common import sha256\n    if prior['signature'] != signature or prior['config_sha256'] != sha256(root / sys.argv[2]):\n        raise ValueError('Target configuration/code/data changed; new version required')\nc = pd.read_csv(root / cfg['canonical_csv'], dtype={'site_id': str})\nif c.date.max() > '2023-12-31': raise ValueError('Final-period rows in canonical')\nif c.loc[c.site_id.isin(cfg['source_ids']), 'date'].max() > '2021-12-31':\n    raise ValueError('Source model period exceeded')\nprint(json.dumps({'signature': signature, 'provenance': provenance}))\n"
    result = subprocess.run([sys.executable, '-c', snippet, str(root), str(path.relative_to(root))],
                            cwd=root, env=env, text=True, capture_output=True)
    if result.returncode:
        raise ValueError(result.stderr.strip() or result.stdout.strip())
    core = json.loads(result.stdout)
    return {'status': 'passed', 'config': str(path.relative_to(root)),
            'config_sha256': digest(path), 'runtime_versions': versions,
            'python': sys.version, 'signature': core['signature'],
            'source_ids': cfg['source_ids'], 'development_ids': cfg['development_ids'],
            'expected_jobs': len(cfg['development_ids']) * len(cfg['origins']) *
                             len(cfg['budgets']) * len(cfg['horizons']) * (1 + len(cfg['seeds'])),
            'physical_facility_independence_verified': False,
            'online_QA_availability_verified': False}


def run_logged(root, config_name, seeds=None, max_jobs=None, strict_versions=True):
    root = Path(root).resolve()
    logs = root / 'runs' / 'colab_logs'
    logs.mkdir(parents=True, exist_ok=True)
    key = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log_path, record_path = logs / (key + '.log'), logs / (key + '.json')
    record = {'status': 'starting', 'config': config_name, 'seeds': seeds,
              'max_jobs': max_jobs, 'started_utc': key, 'scientific_aggregate': False}
    def save():
        temporary = record_path.with_suffix('.tmp')
        temporary.write_text(json.dumps(record, indent=2, ensure_ascii=False))
        temporary.replace(record_path)
    save()
    process = None
    try:
        record['preflight'] = preflight(root, config_name, strict_versions)
        command = [sys.executable, '-u', '-m', 'wastewater.shared_experiment', '--config', config_name]
        if seeds:
            command += ['--seeds', *map(str, seeds)]
        if max_jobs is not None:
            if not isinstance(max_jobs, int) or max_jobs < 1:
                raise ValueError('max_jobs must be a positive integer')
            command += ['--max-jobs', str(max_jobs)]
        record['command'] = command
        record['status'] = 'running'
        save()
        with log_path.open('w') as log:
            process = subprocess.Popen(command, cwd=root, env=project_env(root), text=True,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
            for line in process.stdout:
                log.write(line); log.flush(); print(line, end='', flush=True)
            process.wait()
        record['returncode'] = process.returncode
        record['status'] = 'completed_call' if process.returncode == 0 else 'failed'
        cfg = json.loads(inside(root, config_name).read_text())
        manifest = root / cfg['output'] / 'manifest.json'
        if manifest.exists():
            latest = json.loads(manifest.read_text())
            record['experiment_state'] = {k: latest.get(k) for k in
                ['status', 'completed_jobs', 'expected_jobs', 'failed_jobs', 'counts_this_call']}
        record['log_sha256'] = digest(log_path)
        save()
        if process.returncode:
            raise RuntimeError(f'Experiment failed; retained log: {log_path}')
        return record
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait(timeout=5)
        record['status'] = 'failed_or_interrupted'
        record['error'] = traceback.format_exc()
        with log_path.open('a') as log:
            log.write(record['error'])
        record['log_sha256'] = digest(log_path)
        save()
        raise


def inspect_status(root, config_name):
    root = Path(root).resolve()
    cfg = json.loads(inside(root, config_name).read_text())
    folder = root / cfg['output']
    rows = []
    for path in sorted((folder / 'jobs').glob('*/status.json')):
        status = json.loads(path.read_text())
        if status['status'] == 'completed':
            for name, checksum in status['outputs_sha256'].items():
                if digest(path.parent / name) != checksum:
                    raise ValueError(f'Changed completed artifact: {path.parent / name}')
        rows.append({'task': path.parent.name, 'status': status['status'],
                     'attempt': status.get('attempt'), 'error': status.get('error')})
    expected = len(cfg['development_ids']) * len(cfg['origins']) * len(cfg['budgets']) * \
               len(cfg['horizons']) * (1 + len(cfg['seeds']))
    counts = dict(collections.Counter(row['status'] for row in rows))
    manifest_path = folder / 'manifest.json'
    manifest_status = json.loads(manifest_path.read_text()).get('status') if manifest_path.exists() else 'not_started'
    complete = counts.get('completed', 0) == expected and manifest_status == 'completed'
    return {'status': 'completed' if complete else 'partial_or_not_started',
            'expected_jobs': expected, 'counts': counts, 'manifest_status': manifest_status,
            'uncompleted_tasks': [r for r in rows if r['status'] != 'completed']}


In [ ]:
for archive in ZIP_PATHS:
    print(import_zip(archive, ROOT, EXPECTED_ZIP_SHA256.get(Path(archive).name)))


## 依赖对齐

原实验 CPU PyTorch 2.8.0+cpu。Colab 预装版本可能不同；默认严格拒绝版本差异。下方仅在 `INSTALL_PINNED=True` 时安装最小依赖。升级 torch 后原签名不能续跑，须独立新版本输出。

In [ ]:
if INSTALL_PINNED:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--index-url',
                           'https://download.pytorch.org/whl/cpu', 'torch==2.8.0'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'numpy==2.3.5', 'pandas==2.2.3', 'scipy==1.17.0',
                           'scikit-learn==1.8.0', 'matplotlib==3.10.8'])
    print('安装完成；如已导入旧依赖，请重启 Colab 运行时后从设置单元重新执行。')


In [ ]:
SMOKE_CONFIG = 'configs/shared_v3_20261009/smoke.json'
FULL_CONFIG = 'configs/shared_v3_20261009/full.json'
print(json.dumps(preflight(ROOT, SMOKE_CONFIG, STRICT_VERSIONS), indent=2, ensure_ascii=False))
print(json.dumps(preflight(ROOT, FULL_CONFIG, STRICT_VERSIONS), indent=2, ensure_ascii=False))


## 两个既定测点的完整流程检查

必须先完成原 13013/13015 两个 SiteID。重复执行相同配置将校验缓存；partial 不等于完成。调试接口可临时设置 `max_jobs=2`，完成检查时恢复 `None`。

In [ ]:
smoke_call = run_logged(ROOT, SMOKE_CONFIG, seeds=[20261009], max_jobs=None,
                        strict_versions=STRICT_VERSIONS)
print(json.dumps(inspect_status(ROOT, SMOKE_CONFIG), indent=2, ensure_ascii=False))


## 10 个开发测点：先一个种子，再补两个固定种子

共享训练/选参仅源数据，目标保持四个开发窗口。实际完成状态保存于 Drive 项目。连接中断后重新执行相同单元，已有 GRU epoch/checkpoint 恢复；失败日志保留。

In [ ]:
assert inspect_status(ROOT, SMOKE_CONFIG)['status'] == 'completed', '先完成两个测点检查'
first_seed_call = run_logged(ROOT, FULL_CONFIG, seeds=[20261009],
                             strict_versions=STRICT_VERSIONS)
print(json.dumps(inspect_status(ROOT, FULL_CONFIG), indent=2, ensure_ascii=False))


In [ ]:
# 再次包含首种子，以缓存其目标任务并保留三种子来源训练诊断。
remaining_seeds_call = run_logged(ROOT, FULL_CONFIG, seeds=[20261009, 20261010, 20261011],
                                  strict_versions=STRICT_VERSIONS)
print(json.dumps(inspect_status(ROOT, FULL_CONFIG), indent=2, ensure_ascii=False))


## 保存预测核查与统计报告

只有完整三种子任务完成后生成主要统计报告。以下步骤读取预测，不重新训练。输出报告使用新时间戳目录，既有报告不会覆盖。科学汇总以 SiteID 等权，重复种子取误差均值；不把重复条件当独立水厂。

In [ ]:
state = inspect_status(ROOT, FULL_CONFIG)
assert state['status'] == 'completed', state
full_cfg = json.loads((ROOT / FULL_CONFIG).read_text())
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
validation_file = f'validation/colab_full_{stamp}.json'
(ROOT / 'validation').mkdir(exist_ok=True)
subprocess.check_call([sys.executable, 'scripts/validate_stage2.py', '--run', full_cfg['output'],
                       '--output', validation_file], cwd=ROOT, env=project_env(ROOT))
report_dir = f'reports/colab_shared_v3_{stamp}'
subprocess.check_call([sys.executable, 'scripts/summarize_shared_v3.py', '--run', full_cfg['output'],
                       '--output', report_dir], cwd=ROOT, env=project_env(ROOT))
print('已核查结果：', ROOT / validation_file)
print('逐条预测、逐SiteID指标与区间估计：', ROOT / report_dir)


## 输出解释

主分析保留负预测；统一 `max(0, prediction)` 只作全部方法的次分析。高体积阈值来自每个起点允许的最短 30 天历史 Q90，不是容量、许可或溢流阈值。失败 SiteID 和负迁移不删除。置信区间为有限开发 SiteID 上的探索性时间/站点块估计，不证明独立物理水厂泛化。

当前 notebook 不执行新的模型组合、区间预测、最终测试或论文重写；下一阶段应依据已保存真实点预测结果决定。